<a href="https://colab.research.google.com/github/KunalAyush1/Transformers-Implementation/blob/main/KV_caching_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import numpy as np
import torch

In [76]:
#Assume we have A sunset is --> extremely beautiful
token_1 = (torch.randn(1,4))
token_2 = (torch.randn(1,4))

x = torch.stack((token_1, token_2), dim=0)
x = x.squeeze(1)

print(x.shape)



torch.Size([2, 4])


In [77]:
Wq = (torch.randn(4,4))
Wk = (torch.randn(4,4))
Wv = (torch.randn(4,4))

In [78]:
def prefill(x,Wq,Wk,Wv):
  Q = x @ Wq
  print(f"Q shape: {Q.shape}")
  K = x @ Wk
  print(f"K shape: {K.shape}")
  V = x @ Wv
  print(f"V shape: {V.shape}")

  d_k = K.shape[-1]



  attention_weight = Q @ K.transpose(-2,-1)
  print(f"attention_weight shape: {attention_weight.shape}")
  attention_score = torch.softmax(attention_weight / (d_k ** 0.5), dim=-1)

  context_vector = attention_score @ V
  print(f"Shape of Context vector is {context_vector.shape}")

  return K, V, context_vector


In [80]:
K_cache, V_cache, context_vector = prefill(x,Wq,Wk,Wv)


Q shape: torch.Size([2, 4])
K shape: torch.Size([2, 4])
V shape: torch.Size([2, 4])
attention_weight shape: torch.Size([2, 2])
Shape of Context vector is torch.Size([2, 4])


In [81]:
def decode_next_token(x_new,K_cache, V_cache, Wq, Wk, Wv):
   q = x_new @ Wq
   k = x_new @ Wk
   v = x_new @ Wv

   #KV caching
   K_cache = torch.cat((K_cache, k), dim=0)
   V_cache = torch.cat((V_cache, v), dim=0)

   d_k = K_cache.shape[-1]
   new_attention_weight = q @ K_cache.transpose(-2,-1)
   new_attention_score = torch.softmax(new_attention_weight / (d_k ** 0.5), dim=-1)

   context_vector_new = new_attention_score @ V_cache

   print(f"Shape of last context vector is {context_vector_new.shape}")

   return K_cache, V_cache, context_vector_new


In [82]:
x_new = (torch.randn(1,4))

In [83]:
K_cache, V_cache, context_new = decode_next_token(x_new, K_cache, V_cache, Wq, Wk, Wv)

Shape of last context vector is torch.Size([1, 4])


In [84]:
K_cache.shape

torch.Size([3, 4])

In [86]:
V_cache.shape

torch.Size([3, 4])